# Phase 4 — Model Evaluation

This notebook evaluates the three final models from Phase 3:

1. `condition_best_model.pkl`
2. `severity_best_model.pkl`
3. `treatment_best_model.pkl`

It loads processed test data, generates predictions, calculates metrics, creates confusion matrices, saves reports, and prepares outputs for the next phase.

## Step 1 — Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    log_loss,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import learning_curve

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## Step 2 — Define Project Paths

In [ ]:
ROOT_DIR = Path("..").resolve()

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
EVALUATION_DIR = REPORTS_DIR / "evaluation"

for directory in [REPORTS_DIR, FIGURES_DIR, EVALUATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

## Step 3 — Define Target Configuration

In [ ]:
TARGET_CONFIG = {
    "condition": {
        "display_name": "Mental Health Condition",
        "model_file": "condition_best_model.pkl",
        "processed_folder": "condition",
    },
    "severity": {
        "display_name": "Severity Level",
        "model_file": "severity_best_model.pkl",
        "processed_folder": "severity",
    },
    "treatment": {
        "display_name": "Recommended Treatment",
        "model_file": "treatment_best_model.pkl",
        "processed_folder": "treatment",
    },
}

TARGET_CONFIG

## Step 4 — Helper Function: Load Processed Test Data

In [ ]:
def load_processed_target_data(target_key):
    target_dir = PROCESSED_DIR / TARGET_CONFIG[target_key]["processed_folder"]

    files = {
        "X_train": target_dir / "X_train.csv",
        "X_test": target_dir / "X_test.csv",
        "y_train": target_dir / "y_train.csv",
        "y_test": target_dir / "y_test.csv",
    }

    missing_files = [str(path) for path in files.values() if not path.exists()]
    if missing_files:
        raise FileNotFoundError(
            "Phase 2 processed files not found. Please run notebooks/02_data_preprocessing.ipynb first.\n\n"
            "Missing files:\n" + "\n".join(missing_files)
        )

    X_train = pd.read_csv(files["X_train"])
    X_test = pd.read_csv(files["X_test"])
    y_train = pd.read_csv(files["y_train"]).squeeze("columns")
    y_test = pd.read_csv(files["y_test"]).squeeze("columns")

    return X_train, X_test, y_train, y_test

## Step 5 — Helper Function: Load Trained Model

In [ ]:
def load_best_model(target_key):
    model_path = MODELS_DIR / TARGET_CONFIG[target_key]["model_file"]

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model file not found: {model_path}\n"
            "Please run notebooks/03_model_training.ipynb first."
        )

    model = joblib.load(model_path)
    return model

## Step 6 — Check Required Files

In [ ]:
required_files = []

for target_key, cfg in TARGET_CONFIG.items():
    target_dir = PROCESSED_DIR / cfg["processed_folder"]
    required_files.extend([
        MODELS_DIR / cfg["model_file"],
        target_dir / "X_train.csv",
        target_dir / "X_test.csv",
        target_dir / "y_train.csv",
        target_dir / "y_test.csv",
    ])

file_check = pd.DataFrame({
    "file": [str(path) for path in required_files],
    "exists": [path.exists() for path in required_files],
})

file_check

## Step 7 — Load All Models and Data

In [ ]:
project_data = {}

for target_key in TARGET_CONFIG:
    print(f"Loading data and model for: {target_key}")

    X_train, X_test, y_train, y_test = load_processed_target_data(target_key)
    model = load_best_model(target_key)

    project_data[target_key] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "model": model,
    }

    print("  X_train:", X_train.shape)
    print("  X_test :", X_test.shape)
    print("  y_train:", y_train.shape)
    print("  y_test :", y_test.shape)

## Step 8 — Helper Functions: Metrics

In [ ]:
def safe_predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)
        except Exception:
            return None
    return None


def calculate_multiclass_roc_auc(y_true, y_proba, classes):
    try:
        if y_proba is None:
            return np.nan

        if len(classes) == 2:
            return roc_auc_score(y_true, y_proba[:, 1])

        y_true_bin = label_binarize(y_true, classes=classes)
        return roc_auc_score(y_true_bin, y_proba, average="weighted", multi_class="ovr")
    except Exception:
        return np.nan


def calculate_log_loss(y_true, y_proba, classes):
    try:
        if y_proba is None:
            return np.nan
        return log_loss(y_true, y_proba, labels=classes)
    except Exception:
        return np.nan


def evaluate_model(target_key, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = safe_predict_proba(model, X_test)

    classes = np.array(sorted(pd.Series(y_test).unique()))

    metrics = {
        "target": target_key,
        "display_name": TARGET_CONFIG[target_key]["display_name"],
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_test, y_pred),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "roc_auc_weighted_ovr": calculate_multiclass_roc_auc(y_test, y_proba, classes),
        "log_loss": calculate_log_loss(y_test, y_proba, classes),
    }

    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).T

    predictions_df = pd.DataFrame({
        "actual": y_test.reset_index(drop=True),
        "predicted": pd.Series(y_pred).reset_index(drop=True),
        "is_correct": (pd.Series(y_test).reset_index(drop=True) == pd.Series(y_pred).reset_index(drop=True)),
    })

    return metrics, predictions_df, report_df, y_pred, y_proba

## Step 9 — Evaluate All Three Models

In [ ]:
evaluation_results = []
classification_reports = {}
prediction_tables = {}
prediction_arrays = {}
probability_arrays = {}

for target_key, data in project_data.items():
    print(f"Evaluating: {TARGET_CONFIG[target_key]['display_name']}")

    metrics, preds_df, report_df, y_pred, y_proba = evaluate_model(
        target_key=target_key,
        model=data["model"],
        X_test=data["X_test"],
        y_test=data["y_test"],
    )

    evaluation_results.append(metrics)
    classification_reports[target_key] = report_df
    prediction_tables[target_key] = preds_df
    prediction_arrays[target_key] = y_pred
    probability_arrays[target_key] = y_proba

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df

## Step 10 — Save Evaluation Metrics

In [ ]:
evaluation_metrics_path = EVALUATION_DIR / "evaluation_metrics_summary.csv"
evaluation_df.to_csv(evaluation_metrics_path, index=False)
print("Saved:", evaluation_metrics_path)

## Step 11 — Display Important Metrics

In [ ]:
important_metrics = evaluation_df[
    [
        "target",
        "display_name",
        "accuracy",
        "balanced_accuracy",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
        "roc_auc_weighted_ovr",
        "log_loss",
    ]
].copy()

important_metrics

## Step 12 — Plot Model Evaluation Summary

In [ ]:
plot_df = evaluation_df.melt(
    id_vars=["target"],
    value_vars=["accuracy", "balanced_accuracy", "f1_weighted"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x="target", y="score", hue="metric")
plt.title("Phase 4 Evaluation Summary")
plt.ylim(0, 1)
plt.tight_layout()

summary_plot_path = FIGURES_DIR / "phase4_evaluation_summary.png"
plt.savefig(summary_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", summary_plot_path)

## Step 13 — Helper Function: Plot Confusion Matrix

In [ ]:
def plot_confusion_matrix_for_target(target_key):
    y_test = project_data[target_key]["y_test"]
    y_pred = prediction_arrays[target_key]

    labels = sorted(pd.Series(y_test).unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
    )
    plt.title(f"Confusion Matrix — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()

    save_path = FIGURES_DIR / f"{target_key}_confusion_matrix.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return cm, save_path

## Step 14 — Confusion Matrix: Condition Model

In [ ]:
condition_cm, condition_cm_path = plot_confusion_matrix_for_target("condition")
print("Saved:", condition_cm_path)

## Step 15 — Confusion Matrix: Severity Model

In [ ]:
severity_cm, severity_cm_path = plot_confusion_matrix_for_target("severity")
print("Saved:", severity_cm_path)

## Step 16 — Confusion Matrix: Treatment Model

In [ ]:
treatment_cm, treatment_cm_path = plot_confusion_matrix_for_target("treatment")
print("Saved:", treatment_cm_path)

## Step 17 — Save Classification Reports

In [ ]:
for target_key, report_df in classification_reports.items():
    save_path = EVALUATION_DIR / f"{target_key}_classification_report.csv"
    report_df.to_csv(save_path)
    print("Saved:", save_path)

## Step 18 — View Classification Report: Condition

In [ ]:
classification_reports["condition"]

## Step 19 — View Classification Report: Severity

In [ ]:
classification_reports["severity"]

## Step 20 — View Classification Report: Treatment

In [ ]:
classification_reports["treatment"]

## Step 21 — Save Prediction Tables

In [ ]:
for target_key, preds_df in prediction_tables.items():
    save_path = EVALUATION_DIR / f"{target_key}_predictions.csv"
    preds_df.to_csv(save_path, index=False)
    print("Saved:", save_path)

## Step 22 — Error Analysis Function

In [ ]:
def error_analysis(target_key):
    preds_df = prediction_tables[target_key].copy()

    total_errors = (~preds_df["is_correct"]).sum()
    total_rows = len(preds_df)
    error_rate = total_errors / total_rows if total_rows else 0

    class_error = (
        preds_df
        .groupby("actual")
        .agg(
            total_samples=("actual", "count"),
            wrong_predictions=("is_correct", lambda x: (~x).sum()),
            correct_predictions=("is_correct", "sum"),
        )
        .reset_index()
    )
    class_error["error_rate"] = class_error["wrong_predictions"] / class_error["total_samples"]

    misclassified = preds_df[preds_df["is_correct"] == False].copy()
    confusion_pairs = (
        misclassified
        .groupby(["actual", "predicted"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    return {
        "target": target_key,
        "total_rows": total_rows,
        "total_errors": int(total_errors),
        "error_rate": error_rate,
        "class_error": class_error,
        "confusion_pairs": confusion_pairs,
        "misclassified": misclassified,
    }

## Step 23 — Run Error Analysis for All Targets

In [ ]:
error_analysis_results = {}

for target_key in TARGET_CONFIG:
    result = error_analysis(target_key)
    error_analysis_results[target_key] = result

    print("=" * 70)
    print(TARGET_CONFIG[target_key]["display_name"])
    print("Total rows:", result["total_rows"])
    print("Total errors:", result["total_errors"])
    print("Error rate:", round(result["error_rate"], 4))
    print("=" * 70)

## Step 24 — Save Error Analysis Reports

In [ ]:
for target_key, result in error_analysis_results.items():
    class_error_path = EVALUATION_DIR / f"{target_key}_class_error_analysis.csv"
    confusion_pairs_path = EVALUATION_DIR / f"{target_key}_confusion_pairs.csv"
    misclassified_path = EVALUATION_DIR / f"{target_key}_misclassified_samples.csv"

    result["class_error"].to_csv(class_error_path, index=False)
    result["confusion_pairs"].to_csv(confusion_pairs_path, index=False)
    result["misclassified"].to_csv(misclassified_path, index=False)

    print("Saved:", class_error_path)
    print("Saved:", confusion_pairs_path)
    print("Saved:", misclassified_path)

## Step 25 — Visualize Class-Level Error Rate

In [ ]:
for target_key, result in error_analysis_results.items():
    class_error = result["class_error"].copy()

    plt.figure(figsize=(8, 5))
    sns.barplot(data=class_error, x="actual", y="error_rate")
    plt.title(f"Class Error Rate — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Actual Class")
    plt.ylabel("Error Rate")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()

    save_path = FIGURES_DIR / f"{target_key}_class_error_rate.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

## Step 26 — Probability Confidence Analysis

In [ ]:
def probability_confidence_analysis(target_key):
    y_proba = probability_arrays[target_key]
    preds_df = prediction_tables[target_key].copy()

    if y_proba is None:
        print(f"{target_key}: This model does not support predict_proba().")
        return None

    preds_df["prediction_confidence"] = np.max(y_proba, axis=1)

    summary = preds_df.groupby("is_correct")["prediction_confidence"].describe()

    save_path = EVALUATION_DIR / f"{target_key}_prediction_confidence.csv"
    preds_df.to_csv(save_path, index=False)

    return preds_df, summary, save_path


confidence_results = {}

for target_key in TARGET_CONFIG:
    result = probability_confidence_analysis(target_key)
    confidence_results[target_key] = result
    if result is not None:
        preds_conf, summary, save_path = result
        print("=" * 70)
        print(TARGET_CONFIG[target_key]["display_name"])
        print(summary)
        print("Saved:", save_path)

## Step 27 — Plot Prediction Confidence

In [ ]:
for target_key, result in confidence_results.items():
    if result is None:
        continue

    preds_conf, summary, save_path = result

    plt.figure(figsize=(8, 5))
    sns.histplot(
        data=preds_conf,
        x="prediction_confidence",
        hue="is_correct",
        bins=20,
        kde=True,
    )
    plt.title(f"Prediction Confidence — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Prediction Confidence")
    plt.tight_layout()

    fig_path = FIGURES_DIR / f"{target_key}_prediction_confidence.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## Step 28 — Feature Importance Function

In [ ]:
def get_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        coef = model.coef_
        if coef.ndim == 1:
            importance = np.abs(coef)
        else:
            importance = np.mean(np.abs(coef), axis=0)
    else:
        return None

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importance
    }).sort_values("importance", ascending=False)

    return importance_df

## Step 29 — Save and Plot Feature Importance

In [ ]:
feature_importance_results = {}

for target_key, data in project_data.items():
    model = data["model"]
    feature_names = data["X_test"].columns.tolist()

    importance_df = get_feature_importance(model, feature_names)

    if importance_df is None:
        print(f"{target_key}: Feature importance not available for this model.")
        continue

    feature_importance_results[target_key] = importance_df

    save_path = EVALUATION_DIR / f"{target_key}_feature_importance.csv"
    importance_df.to_csv(save_path, index=False)
    print("Saved:", save_path)

    top_n = min(20, len(importance_df))
    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance_df.head(top_n), x="importance", y="feature")
    plt.title(f"Top {top_n} Feature Importance — {TARGET_CONFIG[target_key]['display_name']}")
    plt.tight_layout()

    fig_path = FIGURES_DIR / f"{target_key}_feature_importance.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## Step 30 — Learning Curve Function

In [ ]:
def plot_learning_curve_for_target(target_key, scoring="f1_weighted"):
    data = project_data[target_key]
    model = data["model"]
    X_train = data["X_train"]
    y_train = data["y_train"]

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=3,
        scoring=scoring,
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=-1,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    learning_df = pd.DataFrame({
        "train_size": train_sizes,
        "train_score_mean": train_mean,
        "train_score_std": train_std,
        "validation_score_mean": val_mean,
        "validation_score_std": val_std,
    })

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, marker="o", label="Training score")
    plt.plot(train_sizes, val_mean, marker="o", label="Validation score")

    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)

    plt.title(f"Learning Curve — {TARGET_CONFIG[target_key]['display_name']}")
    plt.xlabel("Training Samples")
    plt.ylabel(scoring)
    plt.legend()
    plt.tight_layout()

    fig_path = FIGURES_DIR / f"{target_key}_learning_curve.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    save_path = EVALUATION_DIR / f"{target_key}_learning_curve.csv"
    learning_df.to_csv(save_path, index=False)

    print("Saved:", fig_path)
    print("Saved:", save_path)

    return learning_df

## Step 31 — Learning Curve: Condition

In [ ]:
condition_learning_df = plot_learning_curve_for_target("condition")
condition_learning_df

## Step 32 — Learning Curve: Severity

In [ ]:
severity_learning_df = plot_learning_curve_for_target("severity")
severity_learning_df

## Step 33 — Learning Curve: Treatment

In [ ]:
treatment_learning_df = plot_learning_curve_for_target("treatment")
treatment_learning_df

## Step 34 — Combined Prediction Table for App Logic

In [ ]:
min_len = min(len(prediction_tables["condition"]), len(prediction_tables["severity"]), len(prediction_tables["treatment"]))

combined_predictions = pd.DataFrame({
    "condition_actual": prediction_tables["condition"]["actual"].iloc[:min_len].values,
    "condition_predicted": prediction_tables["condition"]["predicted"].iloc[:min_len].values,
    "severity_actual": prediction_tables["severity"]["actual"].iloc[:min_len].values,
    "severity_predicted": prediction_tables["severity"]["predicted"].iloc[:min_len].values,
    "treatment_actual": prediction_tables["treatment"]["actual"].iloc[:min_len].values,
    "treatment_predicted": prediction_tables["treatment"]["predicted"].iloc[:min_len].values,
})

combined_path = EVALUATION_DIR / "combined_three_output_predictions.csv"
combined_predictions.to_csv(combined_path, index=False)

print("Saved:", combined_path)
combined_predictions.head(20)

## Step 35 — Final Evaluation Summary

In [ ]:
best_summary = important_metrics.copy()

for col in ["accuracy", "balanced_accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_weighted_ovr", "log_loss"]:
    best_summary[col] = best_summary[col].round(4)

best_summary

## Step 36 — Generate Markdown Evaluation Report

In [ ]:
def df_to_markdown_safe(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report_lines = []
report_lines.append("# Phase 4 Model Evaluation Report\n")
report_lines.append("This report summarizes the evaluation of the three trained models.\n")

report_lines.append("## Evaluation Metrics\n")
report_lines.append(df_to_markdown_safe(best_summary))
report_lines.append("\n")

report_lines.append("## Generated Files\n")
report_lines.append("- `reports/evaluation/evaluation_metrics_summary.csv`")
report_lines.append("- `reports/evaluation/*_classification_report.csv`")
report_lines.append("- `reports/evaluation/*_predictions.csv`")
report_lines.append("- `reports/evaluation/*_class_error_analysis.csv`")
report_lines.append("- `reports/evaluation/*_confusion_pairs.csv`")
report_lines.append("- `reports/evaluation/*_misclassified_samples.csv`")
report_lines.append("- `reports/evaluation/*_feature_importance.csv`")
report_lines.append("- `reports/figures/*_confusion_matrix.png`")
report_lines.append("- `reports/figures/*_learning_curve.png`")
report_lines.append("- `reports/figures/*_feature_importance.png`")
report_lines.append("\n")

report_lines.append("## Notes\n")
report_lines.append("- Accuracy shows overall correctness.")
report_lines.append("- Weighted F1 is useful when classes are imbalanced.")
report_lines.append("- Confusion matrix helps identify which classes are confused.")
report_lines.append("- Learning curve helps check overfitting and underfitting.")
report_lines.append("- Feature importance gives early insight before Phase 5 Explainable AI.\n")

report_path = REPORTS_DIR / "model_evaluation_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:", report_path)

## Step 37 — Final Phase 4 Output Check

In [ ]:
output_files = [
    EVALUATION_DIR / "evaluation_metrics_summary.csv",
    EVALUATION_DIR / "condition_classification_report.csv",
    EVALUATION_DIR / "severity_classification_report.csv",
    EVALUATION_DIR / "treatment_classification_report.csv",
    EVALUATION_DIR / "condition_predictions.csv",
    EVALUATION_DIR / "severity_predictions.csv",
    EVALUATION_DIR / "treatment_predictions.csv",
    REPORTS_DIR / "model_evaluation_report.md",
    FIGURES_DIR / "condition_confusion_matrix.png",
    FIGURES_DIR / "severity_confusion_matrix.png",
    FIGURES_DIR / "treatment_confusion_matrix.png",
]

final_check = pd.DataFrame({
    "output_file": [str(path) for path in output_files],
    "exists": [path.exists() for path in output_files],
})

final_check

## Step 38 — Phase 4 Conclusion

Phase 4 is complete.

You now have:

- Evaluation metrics for all three models
- Confusion matrices
- Classification reports
- Prediction tables
- Error analysis files
- Learning curves
- Feature importance files
- Final evaluation report

Next phase:

**Phase 5 — Explainable AI with SHAP and PDP**